In [ ]:
from datetime import date
today = date.today()
print('Última corrida el',today)

In [ ]:
from astropy.io import fits 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.wcs import utils
import astropy.units as u
import astropy.coordinates as coords
from photutils.aperture import CircularAperture,aperture_photometry
from astropy.io import fits
from scipy.stats import norm
from scipy.optimize import curve_fit
from numpy import asarray as ar,exp
import scipy.integrate as integrate
from cycler import cycler
import matplotlib as mpl
import os

In [ ]:
# defino parametros generales para los graficos que despues puedo modificar individualmente

styling = {
        'lines.linewidth' : 2,
        'font.family' : 'serif',
        'axes.prop_cycle' : cycler('color', ['#EE6666', '#3388BB', '#9988DD',
                 '#EECC55', '#88BB44', '#FFBBBB']),
        'axes.labelsize' : 20,
        'legend.fontsize' : 20,
        "xtick.labelsize": 25,
        "ytick.labelsize": 25,
        'ytick.major.size' : 25,
        'ytick.minor.size' : 25,
        'xtick.major.size' : 25,
        'xtick.minor.size' : 25,
        'xtick.direction' : 'inout',
        'ytick.direction' : 'inout',  
}

mpl.rcParams.update(styling)

In [ ]:
gal = 'ngc1387'

In [ ]:
cntnm = '{}_cont.fits'.format(gal)
hdu = fits.open(cntnm)
plt.imshow(hdu[1].data)

In [ ]:
specl = pd.read_csv('ngc1387_aper_phot.csv')
specr = pd.read_csv('adj/ngc1387adj_aper_phot.csv')
spec = pd.concat((specl,specr),axis=0)
spec = spec.reset_index()
spec = spec.drop(['id'],axis=1)
spec = spec.rename(columns={"index" : "id"})


dat_name = '{}_dat.cat'.format(gal)
bas = pd.read_csv(dat_name)

In [ ]:
#function to generate an arbitrary wl range of MUSE. Full coverage: (3721)*1.25+4699.625
def wl(N, crpix, crdelt, crval):
    """
    N: numero del slice
    crpix: reference slice de inicio
    crdelt: delta lambda por cada slice
    crval: lambda de referencia/inicio
    """
    return (N+1-crpix)*crdelt + crval

CRPIX1 = 1
CRDEL1 = 1.25
CRVAL1 = float(bas['ngc1387.fits'][2])

In [ ]:
CRVAL1

In [ ]:
rothd = pd.read_csv('ngc1387_pne.cat',delim_whitespace=True,names=['ra','dec','x','y','flux','cen'])
rothu = pd.read_csv('adj/ngc1387adj_pne.cat',delim_whitespace=True,names=['ra','dec','x','y','flux','cen'])
roth = pd.concat((rothd,rothu),axis=0)
roth = roth.reset_index()
roth

In [ ]:
len(roth['x'])

np.savetxt('roth_dropped.txt',roth)

In [ ]:
corrected_flux_df = spec.filter(like='_corrected', axis=1)
original_flux_cols = [col for col in spec.columns
                      if col.startswith('aper_sum_bkgsub_')
                      and not col.endswith('_corrected')]

# Now, filter your DataFrame using this new list of column names
flux_df = spec[original_flux_cols]
error_df = spec.filter(like='total_error',axis=1)

#flux and eflux values will correstond to 
n = 10 #star id-1 (-1 because of python)
# flx = spec.values[n, 3::3]
# eflux = spec.values[n, 4::3]
# cflux = spec.values[n, 5::3]
flux = flux_df.values[n]
eflux = error_df.values[n]
cflux = corrected_flux_df.values[n]
slices_to_extract = np.arange(len(flux))
w = wl(N=slices_to_extract, crpix=CRPIX1, crdelt=CRDEL1, crval=CRVAL1) #creates the array of wl to use with the extracted spectrum
min(w)

In [ ]:
w = w/(1+0.004343)
min(w)


In [ ]:
len(cflux)-len(flux)

In [ ]:
# plt.scatter(-2.5*np.log10(flux),-2.5*np.log10(cflux))
c = 0.17
plt.plot(-2.5*np.log10(flux),-2.5*np.log10(cflux)+c,color='black',linewidth=20)
plt.plot(-2.5*np.log10(flux),-2.5*np.log10(flux),color='pink',linewidth=2)

plt.xlim(-5,-4)
plt.ylim(-5,-4)


In [ ]:
plt.figure(figsize=(18,6))

shift = 0.0 #pix/AA

plt.xlim(4550, 9020)
#plt.subplot(2,1,1)
plt.plot(w-shift, flux/np.max(flux), color='k', lw=1)

# plt.plot(w-shift, cflux/np.max(cflux), color='white', lw=1)
plt.fill_between(w-shift, (flux-eflux)/np.max(flux), (flux+eflux)/np.max(flux), color='purple', alpha=0.3)
# plt.vlines(x=5007.8, ymin=0.9, ymax=1.1, color='blue', alpha=0.3)
plt.vlines(x=5007.8, ymin=-0.3, ymax=-1.1, color='blue', alpha=0.3)


plt.ylim(-1, 1.1)
plt.ylabel(r'${\rm Normalized flux}$', size=37)
plt.xlabel(r'${\rm Wavelength\,[\AA]}$', size=37)

plt.tight_layout()

plt.savefig('spectr_full.png')


In [ ]:
plt.figure(figsize=(10,3))

shift = 0 #pix/AA
minn = 5008-1.25*6
maxx = 5008+1.25*6
plt.xlim(minn, maxx)
#plt.subplot(1,1,1)
plt.plot(w-shift, flux/np.max(flux), color='k', lw=1)
plt.fill_between(w-shift, (flux-eflux)/np.max(flux), (flux+eflux)/np.max(flux), color='purple', alpha=0.5)
plt.vlines(x=5006.843, ymin=0, ymax=1.1, color='gray', alpha=0.8)

plt.ylim(-0.4, 1.2)
plt.ylabel(r'${\rm Normalized Flux}$')
plt.xlabel(r'${\rm Wavelength [\AA]}$')

plt.tight_layout()

plt.savefig('short.png')

plt.show()
plt.figure(figsize=(10,3))

#center on OIII
#plt.subplot(2,1,1)
plt.ylim(-0.4, 1)

plt.plot(w-shift, flux/np.max(flux), color='k', lw=1)
plt.fill_between(w-shift, (flux-eflux)/np.max(flux), (flux+eflux)/np.max(flux), color='purple', alpha=0.5)
#plt.vlines(x=5007, ymin=0.0, ymax=1.0, color='gray', alpha=0.8)
#plt.vlines(x=4959, ymin=0.0, ymax=1.0, color='gray', alpha=0.8)

#plt.savefig('spectrfull.png')

plt.xlim(4948, 5095)
#plt.ylim(0, 1000)
plt.ylim(-0.4, 1.2)
plt.ylabel(r'${\rm Normalized Flux}$')
plt.xlabel(r'${\rm Wavelength [\AA]}$')

plt.tight_layout()
plt.savefig('spectr.png')
plt.show()
plt.figure(figsize=(8,3))

# center on Halpha
shift = 0. #pix/AA
#plt.subplot(2,1,1)
plt.ylim(-0.4, 1)

plt.plot(w-shift, flux/np.max(flux), color='k', lw=1)
plt.fill_between(w-shift, (flux-eflux)/np.max(flux), (flux+eflux)/np.max(flux), color='purple', alpha=0.5)
#plt.vlines(x=6564.61, ymin=0.0, ymax=1.0, color='gray', alpha=0.8)
plt.xlim(6500, 6600)
#plt.ylim(0.3, 0.7)
plt.ylabel('Flux', size=17)
plt.xlabel('Wavelength', size=17)

plt.tight_layout()
plt.savefig('spectr_h.png')

plt.show()



In [ ]:
roth['cen'] = roth['cen']/(1+0.004343)

In [ ]:
flux = cflux
mag = np.zeros(len(spec['id']))
mag2 = np.zeros(len(spec['id']))
flu = np.zeros(len(spec['id']))
m_o = np.zeros(len(spec['id']))
emo = np.zeros(len(spec['id']))
m_o2 = np.zeros(len(spec['id']))
flu2 = np.zeros(len(spec['id']))
flu3 = np.zeros(len(spec['id']))
eflu = np.zeros(len(spec['id']))
eflu2 = np.zeros(len(spec['id']))
eflu3 = np.zeros(len(spec['id']))
emag = np.zeros(len(spec['id']))
emag2 = np.zeros(len(spec['id']))
                 
for i in range(len(spec['id'])):
    
    flux = spec.values[i, 3::2]
    
    
    h_al = 6562.819

    ind = np.where(np.abs((w-h_al))<0.625)
    start = ind[0][0]-3
    end = ind[0][0]+3
    h_fil = flux[start:end]
    h_err = eflux[start:end]
                 
    m_h = np.sum(h_fil,axis=0)
    emh = np.sqrt(np.sum(np.power(h_err,2),axis=0))

    oiii = 5006.843

    ind = np.where(np.abs((w-roth['cen'][i]))<0.625)

    start = ind[0][0]-3
    end = ind[0][0]+3
    o_fil = flux[start:end]
    o_err = eflux[start:end]
    
    emo[i] = np.sqrt(np.sum(np.power(o_err,2),axis=0))
    m_o[i] = np.sum(o_fil,axis=0)
    
    
    m_o2[i] = flux[ind[0][0]]


    nii = 6583.460

    ind = np.where(np.abs((w-nii))<0.625)
    start = ind[0][0]-3
    end = ind[0][0]+3
    n_fil = flux[start:end]
    n_err = eflux[start:end]
    
    m_n = np.sum(n_fil,axis=0)
    emn = np.sqrt(np.sum(np.power(n_err,2),axis=0))
    
    sii = 6716.440	

    ind = np.where(np.abs((w-sii))<0.625)
    start = ind[0][0]-3
    end = ind[0][0]+3
    s_fil = flux[start:end]
    s_err = eflux[start:end]
    
    m_s = np.sum(s_fil,axis=0)
    ems = np.sqrt(np.sum(np.power(s_err,2),axis=0))
    
    sii2 = 6730.810	

    ind = np.where(np.abs((w-sii2))<0.625)
    start = ind[0][0]-3
    end = ind[0][0]+3
    s_fil2 = flux[start:end]
    s2_err = eflux[start:end]

    m_s2 = np.sum(s_fil2,axis=0)
    ems2 = np.sqrt(np.sum(np.power(s2_err,2),axis=0))
    
    if(m_o[i]>0):
        mag[i] = -2.5*np.log10(m_o[i]*np.power(10.0,-20))-13.74
        emag[i] = (2.5/np.log(10))*(emo[i]/m_o[i])
        mag2[i] = -2.5*np.log10(m_o2[i]*np.power(10.0,-20))-13.74
        flu[i] = m_o[i]/(m_h+m_n)
        eflu[i] = np.sqrt(np.power(emo[i]/m_o[i],2)+np.power(emh/m_h,2)+np.power(emn/m_n,2))
        flu2[i] = m_h/m_n
        eflu2[i] = np.sqrt(np.power(emh/m_h,2)+np.power(emn/m_n,2))
        flu3[i] = m_h/(m_s+m_s2)
        eflu3[i] = np.sqrt(np.power(emh/m_h,2)+np.power(ems/m_s,2)+np.power(ems2/m_s2,2))
    else:
        mag[i] = 99
        flu[i] = 0
mag

In [ ]:
from scipy import stats
x = mag[(mag<29)&(mag>26)]
y = mag2[(mag<29)&(mag>26)]
# Linear regression model
gradient, intercept, r_value, p_value, slope_std_error = stats.linregress(x, y)
predict_y = gradient * x + intercept
# Associated error
pred_error = y - predict_y
degrees_of_freedom = len(x) - 2
residual_std_error = np.sqrt(np.sum(pred_error**2) / degrees_of_freedom)

In [ ]:
plt.scatter(mag,mag2)
plt.xlabel('mag original')
plt.ylabel('mag slice')
plt.plot(mag, mag + 1, label='Line of best fit')
plt.plot(mag, gradient*mag+intercept, c='black')
plt.ylim(27,32)
plt.xlim(27,31)
#plt.plot(x,x+1.2)
gradient,intercept

In [ ]:
new = np.asarray([roth['x'],roth['y'],roth['ra'],roth['dec'],roth['cen'],mag,m_o])
new = new.transpose()
np.savetxt('catalogo.txt',new)

<span style='color:darkolivegreen;font-weight:bold'> CHECK DISTANCE MODULUS BELOW </span>

In [ ]:
condition = np.log10(flu)>(-0.37*((mag)-(31.43))-1.16)
condition2 = (flu<4)
condition3 = (eflu<2)
condition4 = (flu>0)
mag2 = mag[condition&condition2]
min(mag2)

In [ ]:
len(np.where(~condition2)[0])


In [ ]:
pne = np.asarray([roth['x'][condition&condition2],roth['y'][condition&condition2],roth['ra'][condition&condition2],roth['dec'][condition&condition2],roth['cen'][condition&condition2],mag[condition&condition2],m_o[condition&condition2]])
pne = pne.transpose()
np.savetxt('pne.txt',pne)

In [ ]:
ind = (np.where(~condition2)[0])
nun = len(ind)

In [ ]:
fig, axes = plt.subplots(13, 1, figsize=(18, 35), constrained_layout=True)
# Flatten the 4x4 array of axes into a 1D array for easier indexing in the loop
axes = axes.flatten()
plt.hspace = 0

for i,n in enumerate(ind):
    flux = flux_df.values[n]
    eflux = error_df.values[n]
    cflux = corrected_flux_df.values[n]
    slices_to_extract = np.arange(len(flux))
    w = wl(N=slices_to_extract, crpix=CRPIX1, crdelt=CRDEL1, crval=CRVAL1) #creates the array of wl to use with the extracted spectrum
    w = w/(1+0.004343)
    
    ax = axes[i]

    ax.set_xlim(4848, 5095)

    ax.set_ylim(-0.4, 1)

    ax.plot(w-shift, flux/np.max(flux), color='k', lw=1)
    ax.vlines(x=5008, ymin=0.6, ymax=1.1, color='gray', alpha=0.8)
    ax.vlines(x= 4959, ymin=0.6, ymax=1.1, color='gray', alpha=0.8)
    ax.fill_between(w-shift, (flux-eflux)/np.max(flux), (flux+eflux)/np.max(flux), color='purple', alpha=0.5)
plt.subplots_adjust(wspace=0, hspace=0)
plt.savefig('interlopers2.png')

In [ ]:
pneh = roth[condition&condition2]
pneh = pneh.reset_index()
pneh = pneh.drop('index',axis=1)
pneh = pneh[['x','y','ra','dec']]
new_df = pd.DataFrame(data=(mag[condition&condition2],emag[condition&condition2]),index=['mag','emag'])
new_df = new_df.T
pneh2 = pd.concat((pneh,new_df),axis=1)
pneh2.to_csv('pne_with_header.csv',header=True,index=['X','Y','RA','DEC','MAG','EMAG'])

In [ ]:
pne = np.asarray([roth['ra'][condition&condition2],roth['dec'][condition&condition2],roth['cen'][condition&condition2],mag[condition&condition2],m_o[condition&condition2]])
pne = pne.transpose()
np.savetxt('coords.txt',pne)

In [ ]:
dig = np.asarray([roth['x'],roth['y'],roth['ra'],roth['dec'],roth['cen'],mag,emag,flu,eflu,flu2,flu3])
dig = dig.transpose()
np.savetxt('for_diag.txt',dig)

In [ ]:
fig,ax = plt.subplots(figsize=(8,6))

magarr = np.arange(20,40)


# plt.errorbar(mag[condition3],flu[condition3],xerr=emag[condition3],yerr=eflu[condition3],color='lightgrey',fmt='none')
plt.errorbar(mag,flu,xerr=emag,yerr=eflu,color='lightgrey',fmt='none', alpha=0.6)

plt.scatter(mag,flu,s=40,color='#118ab2')

plt.scatter(mag[condition&condition2],flu[condition&condition2],s=40,color='#ef476f',marker='x')


y = np.power(10,-0.37*((magarr)-(31.43))-1.16)

plt.yscale('log',base=10)
plt.plot(magarr,y,color='#073b4c',linewidth=1)
plt.hlines(4,20,40,color='#073b4c',linewidth=1)
plt.xlim(24,32)
plt.ylim(0.0000001,18)
# plt.xticks(fontsize=40)
# plt.yticks(fontsize=40)
plt.ylim(0.05,29)
plt.ylabel(r'$[OIII]/(H\alpha+[NII])$')
plt.xlabel('$m_{5007}$')
plt.text(24.4,0.1,'NGC 1387',fontsize=20)

plt.savefig('diagnostic-1387.eps',bbox_inches='tight')

In [ ]:
len(pne)

In [ ]:
fig,ax = plt.subplots(figsize=(8,6))

plt.scatter(np.log10(flu3[condition&condition2]),np.log10(flu2[condition&condition2]),c='#223843')


#y = np.power(10,-0.37*((magarr)-(31))-1.16)
#plt.yscale('log',base=10)
#plt.xscale('log',base=10)
plt.hlines(0.3,0.1,10,linestyle='dashed',color='lightgrey')

plt.vlines(0.5,0.1,10,linestyle='dashed',color='lightgrey')
#plt.plot(flu2,flu3,color='#95b7ff')
#plt.hlines(4,26,30,color='#95b7ff')
plt.xlim(-1,1.2)
plt.ylim(-0.6,1)
plt.text(-0.9,-0.5,'NGC 1387',fontsize=20)


plt.ylabel(r'$[H\alpha]/[NII]$')
plt.xlabel(r'$[H\alpha]/[SII]$')
plt.savefig('diagnostic2.png',bbox_inches='tight')

In [ ]:
min(mag2)

In [ ]:
compl = pd.read_csv('completeness/compl_mag',names=['magnitude','detection_completeness'],delim_whitespace=True)


compl['magnitude'] = compl['magnitude']
compl

In [ ]:
from scipy import interpolate
f1 = interpolate.interp1d(compl['magnitude'],compl['detection_completeness'], fill_value='extrapolate')
plt.xlim(25,32)
plt.ylim(0.1,1.5,0.2)
plt.scatter(compl['magnitude'],compl['detection_completeness'],marker='x',s=82)
plt.xlabel('$m_{5007}$')
plt.ylabel('$Completeness fraction$')
plt.plot(compl['magnitude'],f1(compl['magnitude']))
plt.hlines(0.5,25,33,linestyles='dotted',color='black')
plt.vlines(30.1,0,2,linestyles='dashed',color='black')
#plt.savefig('completeness.png',bbox_inches='tight')


CHECK MAGNITUDE HERE

In [ ]:
binw = np.arange(26.2,31,0.4)


histp = list(np.histogram(mag2,bins=binw))

mids = 0.5*(binw[1:] + binw[:-1])

corr = histp[0]
corr = histp[0]/f1(mids)
##corr,mids,f1(mids)
histp[0], corr
fig,ax = plt.subplots(figsize=(12,6))
plt.step(mids,corr,color='#799af1',linestyle='dashed',linewidth=4)
plt.step(mids,histp[0])
#plt.ylim(0,5)
mids,histp[0]

In [ ]:
class PNLF(object):
    """
    This function calculates the PNLF given magnitude data of PNe (organised in the PN tracers class)
    In case color or detection completeness are not known, put 1
    """
    def __init__(self, PNe, galaxy, Ly_alpha, mag_bins, color_completeness,
        detection_completeness, Ntot = 15, normalized = True):
        self.PNe = PNe
        self.galaxy = galaxy
        self.Ly_alpha = Ly_alpha
        self.mag_bins = mag_bins
        self.color_completeness = color_completeness
        self.detection_completeness = detection_completeness
        self.Nsample = len(self.PNe.mag)
        self.normalized = normalized
        self.Ntot = Ntot
        self.r = 1.#self.Nsample/float(self.Ntot)


    @property
    def N_PN(self):
        N_PN, bin_edges = np.histogram(self.PNe.mag, self.mag_bins)
        return N_PN

    @property
    def bin_edges(self):
        N_PN, bin_edges = np.histogram(self.PNe.mag, self.mag_bins)
        return bin_edges

    @property
    def bin_centers(self):
        return  0.5*(self.bin_edges[1:] + self.bin_edges[:-1])
        

    @property
    def bin_width(self):
        return self.bin_edges[1:] - self.bin_edges[:-1]

    @property
    def N_Ly(self):
        interp = interpolate.interp1d(self.Ly_alpha.m_range, self.Ly_alpha.logN)
        return 10**(interp(self.bin_centers))*self.bin_width

    @property
    def N_corr(self):
        return self.N_PN/self.bin_width/self.color_completeness/self.detection_completeness

    @property
    def N_PN_Lya(self):
        Ly_contr = self.r*self.N_Ly
        if self.normalized == True:
            return (self.N_corr - Ly_contr)/self.Nsample
        else:
            return self.N_corr - Ly_contr

    @property
    def error(self):
        err_poiss = self.N_PN/(self.bin_width*self.color_completeness*self.detection_completeness)**2
        err_Ly =  self.bin_width*self.Ly_alpha.bin_width*self.r*self.N_Ly
        if self.normalized == True:
            return np.sqrt(err_poiss )/self.Nsample
        else:
            return np.sqrt(err_poiss )

    @property
    def LF(self):
        return np.log10(self.N_PN)

    @property
    def log_err(self):
        low = np.log10(self.N_PN) - np.log10(self.N_PN - self.error)
        high = np.log10(self.N_PN + self.error) - np.log10(self.N_PN)
        return [low, high]

    
class galaxy(object):
    def __init__(self, RA_center, dec_center, PA, ell, r_ell, distance):
        self.RA_center = coords.Angle(RA_center, unit = u.hourangle)
        self.dec_center = coords.Angle(dec_center, unit = u.deg)
        self.PA = u.Quantity(0.5*np.pi, u.radian) - u.Quantity(PA, u.radian)
        self.ell = ell
        self.r_ell = u.Quantity(r_ell, unit = u.arcsecond)
        self.distance = distance

    @property
    def distance_modulus(self):
        return 5.*np.log10(self.distance) - 5.

def area_ellipse(ell, a_arr):
    A = [np.pi*(1.-ell)*(a_arr[0]**2)]
    for i in range(1,len(a_arr)):
        A.append(np.pi*(1.-ell)*(a_arr[i]**2 - a_arr[i-1]**2))
    return np.array(A)

class PNe(object):
    def __init__(self, ID, RA, dec, mag, vel, vel_err, galaxy):
        self.ID = ID
        self.RA = coords.Angle(RA, unit = u.hourangle)
        self.dec = coords.Angle(dec, unit = u.deg)
        self.mag = mag
        self.vel = vel
        self.vel_err = vel_err
        self.galaxy = galaxy

    @property
    def RA_shifted(self):
        return self.RA - self.galaxy.RA_center

    @property
    def dec_shifted(self):
        return self.dec - self.galaxy.dec_center

    @property
    def r_major(self):
# RA and dec are already shifted wrt center coordinates
# RA, dec --> astropy angles
# output in degrees
        q = 1. - self.galaxy.ell
        x = self.RA_shifted.degree * np.cos(self.galaxy.PA) - self.dec_shifted.degree
        y = self.RA_shifted.degree * np.sin(self.galaxy.PA) + self.dec_shifted.degree 
        return np.sqrt(x**2 + y**2/q**2)

    @property
    def radius(self):
        return np.sqrt(self.RA_shifted.degree**2 + self.dec_shifted.degree**2)

    @property
    def angle(self):
        x = self.RA_shifted.degree * np.cos(self.galaxy.PA) - self.dec_shifted.degree 
        y = self.RA_shifted.degree * np.sin(self.galaxy.PA) + self.dec_shifted.degree 
        angl = coords.Angle(np.arctan2(y,x), unit = u.rad)
        return angl
    

class Ly_alpha(object):
    """
    This is the luminosity function of Ly-alpha emitters at z = 3.1, scaled to our survey area
    and with fluxes converted to F_5007 / m_5007. This function can be interpolated in order
    to determine the number of Ly-alpha emitters per magnitude bin of the PNLF.
     """
    def __init__(self, L_star = 10**42.66, L_range = 10**np.linspace(41, 44, 40),
        alpha = -1.36, phi0 = 1.28e-3, area_conversion = 8.666666666666e58,
        frac_area = 0.75, area = 1.686e5, bin_width = 0.2):
        self.L_star = L_star
        self.L_range = L_range
        self.alpha = alpha
        self.phi0 = phi0
        self.area_conversion = area_conversion
        self.frac_area = frac_area
        self.area = area #u.Mpc
        self.bin_width = 0.2

    def Schechter(self, x):
        return self.phi0*x**self.alpha*np.exp(-x)*self.frac_area*self.area

    @property
    def logN(self):
        return np.log10(self.Schechter(self.L_star/self.L_range))

    def to_m5007(self, flux):
        return -2.5*np.log10(flux) - 13.74

    @property
    def m_range(self):
        return self.to_m5007(self.L_range/self.area_conversion)[::-1]



In [ ]:
gal = galaxy(RA_center =  54.237750    * u.deg, dec_center =   -35.506639 ,
                PA = 60 * u.deg, ell = 0, r_ell = 10, distance = 19.3e3)

ID_number = list(range(len(pne)))
RAJ2000 = pne[:,1]
decJ2000 = pne[:,2]
magOIII = pne[:,3]
velfake = pne[:,4]

ly = Ly_alpha(1)
PNe = PNe(ID = ID_number, RA = RAJ2000 * u.deg, dec = decJ2000 * u.deg, mag = magOIII, vel = velfake, galaxy = gal, vel_err= velfake)

In [ ]:
obj_pn = PNLF(PNe, galaxy, ly, histp[1], 1, f1(histp[1][0:12]), Ntot = 27, normalized = True)

In [ ]:
errores = obj_pn.error
errores


In [ ]:
with open('output.txt', 'w') as file:
    # Iterate through both arrays simultaneously
    for i, j, k, l in zip(histp[0], corr, errores, mids):
        # Write each pair of elements side by side, separated by a space
        file.write(f'{i} {j} {k} {l} \n')


In [ ]:
def PNLF_analytic(m5007, mag, c1):
    return c1*np.exp(-0.302*mag)*np.exp(0.302*m5007)*(1.- np.exp(3*(-4.51 + mag - m5007)))

def PNLF_analytic2(m5007, c1, c2, mag):
    return c1*np.exp(-c2*mag)*np.exp(c2*m5007)*(1.- np.exp(3*(-4.51 + mag - m5007)))


def PNLFfix(m5007,c1):
    mag = 31.41
    return c1*np.exp(-0.302*mag)*np.exp(0.302*m5007)*(1.- np.exp(3*(-4.51 + mag - m5007)))


#histp2 = list(np.histogram(mag2,bins=binw))
cm = f1(binw)
corr = histp[0]/f1(mids)
#mids2 = 0.5*(binw[1:] +binw[:-1])
condition = mids < 29.4
popt,pcov = curve_fit(PNLF_analytic,mids[condition],corr[condition],p0=[30.5,0.8])
popt2,pcov2 = curve_fit(PNLF_analytic2,mids[condition],corr[condition],p0=[20,0.3,30],bounds=([0,0,0],[20,10,100]))
popt3,pcov3 = curve_fit(PNLFfix,mids[condition],corr[condition],p0=[0.8])

print(popt)

In [ ]:
fig,ax = plt.subplots(figsize=(10,6))
ax = plt.gca()
ax.tick_params(width=3)
plt.grid(True)
plt.step(mids,corr,color='#ef476f',linewidth=4)
plt.step(mids,histp[0],color='grey',linestyle='dashed',linewidth=4)

plt.xlim(26.,30)
plt.xlabel('$m_{5007}$',fontsize=30)
plt.ylabel('$N_{PNe}/mag$',fontsize=30)
plt.ylim(1,16)
x0 = np.arange(23,30.1,0.1)
plt.plot(x0,PNLFfix(x0,*popt3),linestyle='dashdot',label='fit',c='#118ab2',linewidth=5)
plt.plot(x0,PNLF_analytic(x0,*popt),'r-',label='fit',c='#073b4c',linewidth=5)


plt.errorbar(mids-0.25, corr, yerr = errores, fmt = 'none', linewidth=10, c='grey')

x0 = np.arange(28.9,38,0.1)
plt.plot(x0,PNLFfix(x0,*popt3),'r:',label='fit',c='#118ab2',linewidth=5)
plt.plot(x0,PNLF_analytic(x0,*popt),'r:',label='fit',c='#073b4c',linewidth=5)

plt.yscale('log')

plt.savefig('pnlf-1387.png',bbox_inches='tight')

In [ ]:
fig,ax = plt.subplots(figsize=(17,8))
plt.grid(True)
plt.step(mids,corr,color='black',linewidth=4,label='Completeness corrected')
plt.step(mids,histp[0],color='#799af1',linestyle='dashed',linewidth=4,label='Raw')

#plt.hist(mag,18,color='#799af1',density=True)
plt.xlim(26,29)
plt.xlabel('$m_{5007}$',fontsize=60)
plt.ylabel('$N_{NPs}/mag$',fontsize=50)
plt.ylim(0.9,35)
x0 = np.arange(26,29,0.1)
#plt.plot(x0,PNLF_analytic2(x0,*popt2),'r-',c='#de735b',linewidth=5,label='Variable slope')
plt.plot(x0,PNLF(x0),'r-',c='grey',linewidth=5,label='Fixed slope')
plt.yticks(fontsize=45)
plt.xticks(fontsize=45)

#plt.errorbar((mids-0.25), histp[0]/compl['detection_completeness'], yerr = errores, fmt = 'none', linewidth=10)

x0 = np.arange(26,29.5,0.1)
#plt.plot(x0,PNLF_analytic2(x0,*popt2),'r:',c='#de735b',linewidth=5)
plt.plot(x0,PNLF_analytic(x0,*popt),'r:',c='grey',linewidth=5)

plt.yscale('log')
plt.legend(mode='expand',ncol=2)
plt.savefig('pnlf2.png',bbox_inches='tight')

In [ ]:
import scipy.integrate as integrate

In [ ]:
def PNLFint(m5007):
    mag = 31.41
    c1 = popt3
    return c1*np.exp(-0.302*mag)*np.exp(0.302*m5007)*(1.- np.exp(3*(-4.51 + mag - m5007)))

In [ ]:
popt3

In [ ]:
D = 19.3*np.power(10,6) 
mu=5.*np.log10(D)-5
print('Distance modulus: ', mu)
mstar = mu - 4.51
print('Apparent bright cutoff: ', mstar)
print('Apparent mstar+2.5: ', mstar+2.5)

In [ ]:
integrate.quad(PNLFint,mstar,mstar+2.5)[0]/integrate.quad(PNLFint,mstar,30.1)[0]

In [ ]:
### Build comprehensive PNe catalog with computed metrics and flags
import pandas as pd
import numpy as np
import astropy.coordinates as coords
import astropy.units as u

# Align all inputs to the shortest available length to avoid mismatches
min_len = min(
    len(spec), len(roth), len(mag), len(emag), len(m_o), len(emo),
    len(flu), len(eflu), len(flu2), len(eflu2), len(flu3), len(eflu3),
    len(condition), len(condition2)
)

spec_s = spec.iloc[:min_len]
roth_s = roth.iloc[:min_len]
mag_s = np.asarray(mag)[:min_len]
emag_s = np.asarray(emag)[:min_len]
m_o_s = np.asarray(m_o)[:min_len]
emo_s = np.asarray(emo)[:min_len]
flu_s = np.asarray(flu)[:min_len]
eflu_s = np.asarray(eflu)[:min_len]
flu2_s = np.asarray(flu2)[:min_len]
eflu2_s = np.asarray(eflu2)[:min_len]
flu3_s = np.asarray(flu3)[:min_len]
eflu3_s = np.asarray(eflu3)[:min_len]
is_pn_s = (np.asarray(condition)[:min_len] & np.asarray(condition2)[:min_len])

# Sexagesimal RA/DEC strings for readability
ra_deg = pd.to_numeric(roth_s['ra'], errors='coerce')
dec_deg = pd.to_numeric(roth_s['dec'], errors='coerce')
ra_angles = coords.Angle(ra_deg.values, unit=u.deg)
dec_angles = coords.Angle(dec_deg.values, unit=u.deg)
ra_hms = ra_angles.to_string(unit=u.hourangle, sep=':', precision=2)
dec_dms = dec_angles.to_string(unit=u.degree, sep=':', precision=2, alwayssign=True)

catalog_df = pd.DataFrame({
    'ID': spec_s['id'],
    'X': pd.to_numeric(roth_s['x'], errors='coerce'),
    'Y': pd.to_numeric(roth_s['y'], errors='coerce'),
    'RA_deg': ra_deg,
    'DEC_deg': dec_deg,
    'RA_hms': ra_hms,
    'DEC_dms': dec_dms,
    'OIII_center_A': pd.to_numeric(roth_s['cen'], errors='coerce'),
    'm5007': mag_s,
    'em5007': emag_s,
    'OIII_flux_sum': m_o_s,
    'eOIII_flux_sum': emo_s,
    'OIII_over_Ha+NII': flu_s,
    'eOIII_over_Ha+NII': eflu_s,
    'Ha_over_NII': flu2_s,
    'eHa_over_NII': eflu2_s,
    'Ha_over_SII': flu3_s,
    'eHa_over_SII': eflu3_s,
    'completeness': f1(mag_s),
    'is_PN': is_pn_s
})

# Sort by brightness (smaller m5007 first) and apply gentle rounding
num_cols = [
    'X','Y','RA_deg','DEC_deg','OIII_center_A','m5007','em5007',
    'OIII_flux_sum','eOIII_flux_sum','OIII_over_Ha+NII','eOIII_over_Ha+NII',
    'Ha_over_NII','eHa_over_NII','Ha_over_SII','eHa_over_SII','completeness'
]
for c in num_cols:
    if c in catalog_df.columns:
        catalog_df[c] = pd.to_numeric(catalog_df[c], errors='coerce')

catalog_df = catalog_df.sort_values('m5007').reset_index(drop=True)
catalog_selected = catalog_df[catalog_df['is_PN']].copy()

# Save outputs
full_out = f"pne_catalog_full_{gal}.csv"
sel_out = f"pne_catalog_selected_{gal}.csv"
catalog_df.to_csv(full_out, index=False)
catalog_selected.to_csv(sel_out, index=False)

print(f"Saved: {full_out} ({len(catalog_df)} rows)")
print(f"Saved: {sel_out} ({len(catalog_selected)} rows)")
print("Preview of selected catalog (top 5):")
try:
    from IPython.display import display
    display(catalog_selected.head())
except Exception:
    print(catalog_selected.head().to_string())